# 📐 04. Pairwise Feature Engineering & Analysis
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

This notebook inspects the engineered pairwise similarity feature matrix:
1. Exact string matches (`exact_name_match`, `exact_address_match`).
2. String distance metrics:
   - Jaro-Winkler similarity on full name and clean name
   - Levenshtein ratio on name and address
3. Token-based overlap metrics:
   - Token Sort Ratio & Token Set Ratio
   - Token Jaccard overlap on business name and address
4. Address component features:
   - Extracted digit match (house/unit numbers)
   - Digits Jaccard similarity
5. Country & missingness indicators:
   - Open-set country match flag (`country_exact_match`)
   - Null indicator flags.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
feat_parquet = PROJECT_ROOT / cfg["paths"]["processed_dir"] / "train_features.parquet"

if feat_parquet.exists():
    df_feat = pd.read_parquet(feat_parquet)
    print(f"Feature matrix shape: {df_feat.shape}")
    display(df_feat.head(5))
else:
    print(f"Feature matrix not found at {feat_parquet}.")

## 1. Feature Separation: Positive vs Negative Matches
Compare mean feature values between true matches (`label == 1`) and non-matches (`label == 0`).

In [ ]:
if feat_parquet.exists():
    feature_cols = [c for c in df_feat.columns if c not in ["source1_entity_id", "candidate_entity_id", "label"]]
    stats = df_feat.groupby("label")[feature_cols].mean().T
    stats.columns = ["Non-Match (0)", "True Match (1)"]
    stats["Difference"] = stats["True Match (1)"] - stats["Non-Match (0)"]
    display(stats.sort_values(by="Difference", ascending=False))